# Collected edits, OOD generality

All three dermatologists go into one codebook per model, one row per case. A case marked wrong by anyone is edited; if the corrections disagree it is printed and dropped. Reliability and locality stay in the perturb_generality collect run. This notebook scores leftover combined MIDAS cases: saved top-1 against `y16`, then one edited generation against the same `y16`.

`gen_all before` is the saved prediction against `y16`. `gen_all after` is the edited generation against the same `y16`. `gen_acc` is how many originally wrong cases become correct. `gen_loc` is how many originally correct cases stay correct.

`dermedit` kernel. `OVERWRITE` recomputes a saved run. `RESUME` finishes a run that is still missing generality. With both off, a complete result is printed from disk and an unfinished one stays blank. One embedding mode and one `REJECT_THRESHOLD` per session. `run.py` runs both embedding modes at 1 and at 5.

In [1]:
import sys
from pathlib import Path

OOD = Path("/scratch/jq2uw/MME/derm_vlm_edit/editing/ood_generality")
sys.path.insert(0, str(OOD))
from utils.bootstrap import bootstrap, results_dir
bootstrap()

import config as C

USERS = [
    "jesse.y.liu.md@gmail.com",
    "achiou@stanford.edu",
    "lhunterellul",
]
MODELS = ["medgemma", "dermatollama"]
RATIONALE_MODE = C.RATIONALE_MODE
NORM_ANSWER = "y16"
SUBSAMPLE = 0
MAX_NEW_TOKENS = C.MAX_NEW_TOKENS
SEED = 42
OVERWRITE = False
RESUME = False

# EMBED_MODE:
#   standalone_dual  DINOv2 image concatenated with SBERT text. This is the current run.
#   dual_sbert       VLM vision activation concatenated with SBERT. Still dual.
#   vision           VLM vision activation only.
#   language         A middle language-layer activation only.
#   language_last    Late language-layer activation. The old VLM-key style.
EMBED_MODE = "standalone_dual"
EMBED_MODES = ["standalone_dual", "language_last"]
# The table lists every percentile. REJECT_THRESHOLD is the one this session computes.
THRESHOLDS = [0.1, 1, 5, 10]
REJECT_THRESHOLD = 1
IMAGE_ENCODER = "dinov2"
IMAGE_POOL = "cls"
TEXT_ENCODER = "sbert"
SCALER_DATASET = "coco"

OUT = results_dir(OOD, "collect", str(EMBED_MODE) + "_" + str(REJECT_THRESHOLD))
POOL = "all"
print("results:", OUT)

results: /sfs/weka/scratch/jq2uw/MME/derm_vlm_edit/results/ood_gen/standalone_dual_1/collect


In [2]:
import gc

from IPython.display import Markdown, display

from utils.edit import edit
from utils.edits import collect_all, collect_ood
from utils.eval import load_suite, load_vlm, needs_compute, render_table, run_suite

rows = []
for model_short in MODELS:
    if not needs_compute(OUT, POOL, model_short, overwrite=OVERWRITE, resume=RESUME):
        print(f"{POOL} / {model_short}: loaded")
        rows.append(load_suite(OUT, POOL, model_short))
        continue

    edits, report = collect_all(
        USERS, model_short, rationale_mode=RATIONALE_MODE,
        norm_answer=NORM_ANSWER, subsample=SUBSAMPLE,
    )
    print(f"{model_short}: {len(edits)} unique edits, {len(report['conflicts'])} conflicting cases dropped")
    for row in report["conflicts"]:
        users = ", ".join(str(u).split("@")[0] for u in row["users"])
        print(f"  drop {row['case_id']}: {users} -> {', '.join(row['answers'])}")
    ood = collect_ood(USERS, model_short, subsample=SUBSAMPLE)
    n_before_text = sum(1 for item in ood if item.get("pred_before"))
    print(f"{model_short}: {len(ood)} ood cases ({n_before_text} with a saved top-1)")
    if not edits:
        print(f"skip {model_short}")
        continue

    import torch
    vlm, config = load_vlm(
        model_short, max_new_tokens=MAX_NEW_TOKENS, seed=SEED,
        embed_mode=EMBED_MODE, image_encoder=IMAGE_ENCODER,
        image_pool=IMAGE_POOL, text_encoder=TEXT_ENCODER,
        scaler_dataset=SCALER_DATASET, reject_threshold=REJECT_THRESHOLD,
    )
    editor, queries = edit(vlm, config, edits)
    rows.append(run_suite(
        editor, vlm, queries, user=POOL, model_short=model_short,
        out_dir=OUT, ood_rows=ood,
        norm_answer=NORM_ANSWER, max_new_tokens=MAX_NEW_TOKENS,
        overwrite=OVERWRITE,
    ))
    del vlm, editor
    gc.collect()
    torch.cuda.empty_cache()

saved = []
for mode in EMBED_MODES:
    for pct in THRESHOLDS:
        out = results_dir(OOD, "collect", f"{mode}_{pct:g}")
        for model_short in MODELS:
            row = load_suite(out, POOL, model_short)
            row["embed_mode"] = mode
            row["threshold"] = f"{pct:g}"
            saved.append(row)
display(Markdown(render_table(saved)))

all / medgemma: loaded
all / dermatollama: loaded


| Mode | Threshold | Model | gen_all before | gen_all after | gen_acc | gen_loc |
| --- | --- | --- | --- | --- | --- | --- |
| standalone_dual | 0.1 | MedGemma | 263/746 | 183/746 | 38/483 | 145/263 |
| standalone_dual | 0.1 | DermatoLlama | 263/743 | 172/743 | 37/480 | 135/263 |
| standalone_dual | 1 | MedGemma | 263/746 | 183/746 | 38/483 | 145/263 |
| standalone_dual | 1 | DermatoLlama | 263/743 | 172/743 | 37/480 | 135/263 |
| standalone_dual | 5 | MedGemma | 263/746 | 137/746 | 47/483 | 90/263 |
| standalone_dual | 5 | DermatoLlama | 263/743 | 161/743 | 42/480 | 119/263 |
| standalone_dual | 10 | MedGemma | 263/746 | 106/746 | 55/483 | 51/263 |
| standalone_dual | 10 | DermatoLlama | 263/743 | 130/743 | 49/480 | 81/263 |
| language_last | 0.1 | MedGemma | 263/746 | 173/746 | 39/483 | 134/263 |
| language_last | 0.1 | DermatoLlama | -- | -- | -- | -- |
| language_last | 1 | MedGemma | 263/746 | 173/746 | 39/483 | 134/263 |
| language_last | 1 | DermatoLlama | -- | -- | -- | -- |
| language_last | 5 | MedGemma | 263/746 | 131/746 | 47/483 | 84/263 |
| language_last | 5 | DermatoLlama | -- | -- | -- | -- |
| language_last | 10 | MedGemma | 263/746 | 119/746 | 54/483 | 65/263 |
| language_last | 10 | DermatoLlama | -- | -- | -- | -- |

## Which edit was retrieved

Left is an unseen leftover case. Right is the codebook image retrieval returned for it. The cell rebuilds the codebook for `INSPECT_MODEL` at the `EMBED_MODE` and `REJECT_THRESHOLD` above. It prefers cases the saved run flipped from correct to wrong, then cases it fixed.

In [ ]:
import json

import matplotlib.pyplot as plt
from PIL import Image

from utils.edit import edit
from utils.edits import collect_all, collect_ood
from utils.eval import load_vlm, result_path

INSPECT_MODEL = "medgemma"
N_SHOW = 4

saved_path = result_path(OUT, "generality", POOL, INSPECT_MODEL)
saved_cases = json.loads(saved_path.read_text())["cases"]
flipped = [c for c in saved_cases if c["retrieved"] and c["correct_before"] and not c["correct"]]
fixed = [c for c in saved_cases if c["retrieved"] and not c["correct_before"] and c["correct"]]
picked = (flipped[: N_SHOW // 2] + fixed[: N_SHOW - N_SHOW // 2]) or [
    c for c in saved_cases if c["retrieved"]
][:N_SHOW]

edits, _ = collect_all(
    USERS, INSPECT_MODEL, rationale_mode=RATIONALE_MODE, norm_answer=NORM_ANSWER,
)
import torch
vlm, config = load_vlm(
    INSPECT_MODEL, max_new_tokens=MAX_NEW_TOKENS, seed=SEED,
    embed_mode=EMBED_MODE, image_encoder=IMAGE_ENCODER,
    image_pool=IMAGE_POOL, text_encoder=TEXT_ENCODER,
    scaler_dataset=SCALER_DATASET, reject_threshold=REJECT_THRESHOLD,
)
editor, queries = edit(vlm, config, edits)
by_case = {row["case_id"]: row for row in collect_ood(USERS, INSPECT_MODEL)}

fig, axes = plt.subplots(len(picked), 2, figsize=(8, 3.2 * len(picked)))
if len(picked) == 1:
    axes = [axes]
for ax, case in zip(axes, picked):
    src = by_case[case["case_id"]]
    matched = editor._get_matched_indices(src["image"], C.QUESTION)
    left, right = ax
    left.imshow(Image.open(src["image"]).convert("RGB"))
    left.set_title(
        f"OOD {case['case_id']}\ntrue {case['y16']}\n"
        f"before {case['pred_before']}\nafter {case['pred']}",
        fontsize=8,
    )
    left.axis("off")
    if not matched:
        right.axis("off")
        right.set_title("no edit retrieved", fontsize=8)
        continue
    hit = queries.data[next(iter(matched))]
    right.imshow(Image.open(hit["image"]).convert("RGB"))
    right.set_title(
        f"edit {hit.get('case_id')}\ncorrection {hit.get('answer')}\n"
        f"y16 {hit.get('human_top1_y16')}",
        fontsize=8,
    )
    right.axis("off")
fig.tight_layout()
plt.show()
del vlm, editor
torch.cuda.empty_cache()